# Jaw Keypoint Training

Notebook wrapper around [`train.py`](train.py). Edit the config cell below, then run all cells.

Requires `../data/train.pkl` and `../data/val.pkl` (from `Create Dataset/create_dataset.py`).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch

sys.path.insert(0, str(Path.cwd()))
from train import default_training_config, run_training

In [ ]:
# ── Training config (same options as train.py CLI) ─────────────────────────
config = default_training_config(
    train_pkl="../data/train.pkl",
    val_pkl="../data/val.pkl",
    out_dir="./checkpoints",
    epochs=80,
    batch_size=8,
    lr=1e-3,
    weight_decay=1e-4,
    num_workers=4,
    img_h=240,
    img_w=320,
    window_size=8,
    temporal_hidden=384,
    temporal_layers=3,
    decoder_hidden=512,
    edge_mode="pad",
    no_require_consecutive=False,
    no_freeze_backbone=False,
    unfreeze_backbone_epoch=0,  # e.g. 20 to fine-tune backbone from epoch 20
    patience=15,
    seed=42,
)

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"Train: {config.train_pkl}")
print(f"Val:   {config.val_pkl}")
print(f"Out:   {config.out_dir}")

In [ ]:
result = run_training(config)

In [ ]:
history = result["history"]
epochs = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history["train_loss"], label="train")
axes[0].plot(epochs, history["val_loss"], label="val")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].legend()
axes[0].set_title("Heatmap MSE loss")

axes[1].plot(epochs, history["val_pck_mean"], label="PCK@10px (mean)")
axes[1].plot(epochs, history["val_rmse_mean"], label="RMSE px (mean)")
axes[1].set_xlabel("epoch")
axes[1].legend()
axes[1].set_title("Validation metrics")

plt.tight_layout()
plt.show()

print(f"Best val loss: {result['best_val_loss']:.4f}")
print(f"Checkpoint: {Path(result['out_dir']) / 'best_model.pt'}")